# Datamine TRIFIL Process Example

This notebook demonstrates how to configure and run the **`trifil`** process wrapper in `dmstudio`.

## Process Description

## TRIFIL

Generates a Datamine cell or subcell model from a triangulated (wireframe) definition, either a solid model or a digital terrain model, with accurate preservation of enclosed volume.

If the input prototype model &**PROTO** is a Rotated Model, as defined using the [PROTOM](<protom.md>) process, then the coordinates of the input points in both the &WIREPT and PERIMIN files must be in the local (rotated) coordinate system. This can be achieved using the [CDTRAN](<cdtran.md>) process. The nine Rotated Model fields (**X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3, ROTAXIS1, ROTAXIS2, ROTAXIS3**) will be copied from the input &**PROTO** file to the output &**MODEL** file.

**Note** : TRIFIL can fill wireframe data that sits outside the Z range of the input block model.

### Input Files:

* **proto** (Block Model Prototype):
  Model prototype file.
  Required=Yes

* **wiretr** (Wireframe Triangle):
  Input wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe Points):
  Input wireframe point file.
  Required=Yes

* **perimin** (String):
  Optional perimeter input file to control area over which model is generated.
  Required=No

### Output Files:

* **model** (Block Model):
  Output model file to be created.
  Required=Yes

### Fields:

* **zone** (Any : MODEL):
  Name of zone field for wireframe with multiple zones to be filled. The field can be
  either numeric or alpha. However if the field is an alpha field then it can contain a
  maximum of 24 characters. If not specified and a field ZONE exists in WIRETR then it
  will automatically be used.
  Default=Undefined
  Required=No

### Parameters:

* **modltype**:
  Type of wireframe model to be filled; one of the following options, with default of (1)
  :- Options: 1: solid 3d, interior to be filled with cells; 2: solid 3d, exterior to be
  filled with cells; 3: surface, cells to be filled below (for XY), to south (for XZ), or
  to west (for YZ); 4: surface, cells to be filled above (for XY), to north (for XZ), or
  to east (for YZ); 5: two surfaces, cells to be fill between.; 6: two surfaces, cells to
  be filled above upper surface and below lower surface.
  Range=1,6
  Values=1,2,3,4,5,6
  Default=1
  Required=Yes

* **zone**:
  Zone code to be inserted into output model **ZONE** field.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxdip**:
  Maximum gradient for any triangle intersecting a cell before splitting into subcells
  (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **splits**:
  Maximum amount of splitting to be allowed (3), within range 0 [for 1 x 1] to 3 [for 8 x
  8].
  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **plane**:
  Optional alpha parameter defining orientation 'XY', 'XZ', or 'YZ', of plane in which
  subcell splitting is to be performed. Please note that care must be taken in selection
  of the plane to be used if the ends of the wireframe have not been linked, as the
  wireframe model is then partially 'hollow' when viewed from certain directions.
  Range=Undefined
  Values='XY', 'XZ', 'YZ'
  Default='XY'
  Required=No

* **xsubcell**:
  Cell division in X direction (1). Max 100.
  Range=1,100
  Values=Undefined
  Default=1
  Required=No

* **ysubcell**:
  Cell division in Y direction (1). Max 100.
  Range=1,100
  Values=Undefined
  Default=1
  Required=No

* **zsubcell**:
  Cell division in Z direction (1). Max 100.
  Range=1,100
  Values=Undefined
  Default=1
  Required=No

* **pvalue**:
  PVALUE of single perimeter to be selected from the PERIMIN file.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **resol**:
  Defines boundary resolution in direction perpendicular to plane of filling. =(0) -
  precise boundary location. = N - boundary rounded to nearest 1/Nth of parent cell size.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **checkrot**:
  Set to 1 to automatically detect and correctly process rotated models. Using this
  parameter means that the input wireframe points file no longer needs to be transformed
  into the model space before using **TRIFIL**.
  Range=0, 1
  Values=Undefined
  Default=0
  Required=No

* **autosort**:
  Set to 1 to automatically sort the output model by IJK if necessary. Using this option
  removes the need to sort the model after running **TRIFIL**. Sorting is generally only
  required when using multiple **ZONE** fields or multiple perimeters, or when the
  direction of filling is not XY. An output message is given at then end of the process
  indicating if the output model may require sorting. =0 : Do not automatically sort the
  output model by IJK. =1 : Automatically sort the output model by IJK.
  Range=0, 1
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('trifil')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute trifil
print("Running trifil...")
dm_cmd.trifil(
    proto_i='t_mod1',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    perimin_i='optional',  # required
    model_o='t_trifil_out',  # required
    modltype_p='required_val',  # required
    # zone_f='optional',  # optional
    # zone_p='optional',  # optional
    # maxdip_p=0,  # optional
    # splits_p=3,  # optional
    # plane_p='XY',  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # pvalue_p='optional',  # optional
    # resol_p=0,  # optional
    # checkrot_p=0,  # optional
    # autosort_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("trifil execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_trifil_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")